# Лекция 2 · Основы метода Монте-Карло

Выполняйте ячейки сверху вниз. Номера и порядок разделов соответствуют слайдам.

Необходимые библиотеки: `numpy matplotlib`.


## Содержание
- [2.1 Основы метода Монте-Карло](#section-2-1)
- [2.2 Интегрирование методом Монте-Карло](#section-2-2)
- [2.3 Случайное блуждание и диффузия](#section-2-3)
- [2.4 Теплоперенос поперёк плёнки](#section-2-4)


<a id="section-2-1"></a>

## 2.1 Основы метода Монте-Карло


### Как получить случайную точку в квадрате


In [3]:
# Независимые x и y равномерно заполняют единичный квадрат.
# 独立均匀抽取 x 和 y，使点均匀分布在单位正方形中。
import numpy as np

rng = np.random.default_rng(42)
x = rng.random(1000)
y = rng.random(1000)

### Оценка \pi по 100 000 точкам


In [4]:
# Доля попаданий в четверть круга приближает π/4.
# 落在四分之一圆内的点所占比例近似为 π/4。
import numpy as np

rng = np.random.default_rng(42)
N = 100000
x = rng.random(N)
y = rng.random(N)
hit = x**2 + y**2 <= 1
pi_mc = 4 * hit.mean()
print(hit.sum(), pi_mc)

78444 3.13776


### Ошибка выборки
Стандартная ошибка доли убывает как N^(-1/2).


In [5]:
# Стандартная ошибка оценки π равна 4√(p(1−p)/n) и убывает как n⁻¹ᐟ².
# π 估计的标准误差为 4√(p(1−p)/n)，按 n 的平方根倒数下降。
sizes = np.array([100, 1000, 10000, 100000])
for n in sizes:
    rng = np.random.default_rng(42)
    hit = rng.random(n)**2 + rng.random(n)**2 <= 1
    p = hit.mean()
    print(n, 4*p, "Стандартная ошибка:", 4*np.sqrt(p*(1-p)/n))

100 3.36 Стандартная ошибка: 0.1466424222385869
1000 3.148 Стандартная ошибка: 0.05178895635171653
10000 3.1488 Стандартная ошибка: 0.016371495227986966
100000 3.13776 Стандартная ошибка: 0.005201444205602901


<a id="section-2-2"></a>

## 2.2 Интегрирование методом Монте-Карло


### Тот же треугольник в Python


In [ ]:
# Интеграл x на [0, 2] оцениваем как длину интервала, умноженную на среднюю высоту.
# 用区间长度乘平均函数值，估计 x 在 [0, 2] 上的积分。
import numpy as np

rng = np.random.default_rng(42)
a, b = 0, 2
N = 100000
x = rng.uniform(a, b, N)
heights = x

I_mc = (b - a) * heights.mean()
print(f"{I_mc:.6f}")

### Как вычислять высоты для другой кривой?


In [ ]:
# Меняем подынтегральную функцию, сохраняя тот же метод среднего значения.
# 更换被积函数，仍使用平均值法计算积分。
import numpy as np

rng = np.random.default_rng(42)
a, b = 0, 1
N = 100000
x = rng.uniform(a, b, N)
heights = np.exp(-x**2)
I_mc = (b - a) * heights.mean()
print(f"{I_mc:.6f}")

### Двумерный интеграл по 200 000 точкам


In [ ]:
# Каждая строка points — точка в единичном квадрате; его площадь равна 1.
# points 的每行是单位正方形中的一个点；区域面积为 1。
import numpy as np

rng = np.random.default_rng(42)
N = 200000
points = rng.random((N, 2))
r2 = np.sum(points**2, axis=1)
heights = np.exp(-r2)
print(f"{heights.mean():.5f}")

### Шесть координат у каждой точки


In [ ]:
# В шести измерениях меняется число координат, но оценка остаётся средним значением.
# 六维积分增加了坐标数，估计量仍为函数值的平均数。
import numpy as np

rng = np.random.default_rng(42)
N = 200000
points = rng.random((N, 6))
r2 = np.sum(points**2, axis=1)
heights = np.exp(-r2)
print(f"{heights.mean():.5f}")

### Эталон для разделимого интеграла


In [ ]:
# Разделимость exp(−Σx²) позволяет получить эталоны I₂ = I₁² и I₆ = I₁⁶.
# exp(−Σx²) 可分离，因此参考值满足 I₂ = I₁²、I₆ = I₁⁶。
from math import erf, sqrt, pi
I1 = sqrt(pi) * erf(1) / 2
print("I1, I2, I6:", I1, I1**2, I1**6)

<a id="section-2-3"></a>

## 2.3 Случайное блуждание и диффузия


### Как смоделировать 20 000 траекторий


In [ ]:
# M — число траекторий, N — число шагов; cumsum суммирует перемещения во времени.
# M 为轨迹数，N 为步数；cumsum 沿时间累加各步位移。
import numpy as np

rng = np.random.default_rng(42)
M, N = 20000, 500
ell, dt = 1.0, 1.0
steps = rng.choice([-ell, ell], size=(M, N))
x = np.cumsum(steps, axis=1)
t = np.arange(1, N + 1) * dt

msd = (x**2).mean(axis=0)
# Используем одномерный закон диффузии ⟨x²⟩ = 2Dt.
# 利用一维扩散关系 ⟨x²⟩ = 2Dt 估计扩散系数。
D_mc = msd[-1] / (2 * t[-1])
print(f"{D_mc:.5f}")

### Траектории, распределение и средний квадрат смещения


In [ ]:
# Сравниваем отдельные траектории, распределение конечных положений и закон ⟨x²⟩ = 2Dt.
# 比较单条轨迹、终点分布与均方位移规律 ⟨x²⟩ = 2Dt。
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(t, x[:6].T)
axes[0].set(xlabel="t", ylabel="x")
axes[1].hist(x[:, -1], bins=30, density=True)
axes[1].set(xlabel="x(t_final)", ylabel="Плотность")
axes[2].plot(t, msd, label="Монте-Карло")
axes[2].plot(t, ell**2/dt*t, "--", label="2Dt")
axes[2].set(xlabel="t", ylabel="Средний квадрат смещения")
axes[2].legend()
plt.tight_layout()
plt.show()

<a id="section-2-4"></a>

## 2.4 Теплоперенос поперёк плёнки


### Параметры и два участка полёта
Серая модель: изотропное рассеяние и потоковое распределение при испускании. Единицы СИ.


In [ ]:
# Основные параметры заданы в СИ; N пакетов выпускается из каждого термостата.
# 主要参数采用国际单位制；每个热库发射 N 个粒子包。
import numpy as np
import matplotlib.pyplot as plt
L = 200e-9
C, v, mean_free_path = 1.6077e6, 6400.0, 43.7e-9
T0, TL, TR = 300.0, 301.0, 299.0
RELAXATION_TIME = mean_free_path/v
N, cells = 50_000, 20   # на каждый резервуар
edges = np.linspace(0, L, cells+1)
dz = L/cells
rng = np.random.default_rng(2026)
print("Время релаксации (пс):", RELAXATION_TIME/1e-12)
z_nm = np.array([160., 176.])
vz_nm_ps = np.array([3.2, 4.8])
wall_time = (200-z_nm)/vz_nm_ps
free_time = np.array([5., 8.])
# Полёт заканчивается при первом событии: рассеянии или достижении стенки.
# 飞行在最先发生的事件处结束：散射或到达边界。
duration = np.minimum(free_time, wall_time)
print("Длительность (пс), выход на границу:", duration, wall_time <= free_time)
print("Конечная координата (нм):", z_nm+vz_nm_ps*duration)
assert np.allclose(duration, [5, 5])

### События и время пребывания
При испускании μ=±√U, после рассеяния μ=2U−1. Время в ячейке равно длине пересечения, делённой на |vμ|.


In [ ]:
# Следим за пакетами до выхода в термостат и накапливаем время пребывания в ячейках.
# 跟踪粒子包直至离开薄膜进入热库，并累加其在各网格中的停留时间。
def histories(side, rng):
    z = np.full(N, 0.0 if side == "left" else L)
    # Испускание с поверхности: |μ| = √U, знак направлен внутрь плёнки.
    # 表面发射采用 |μ| = √U，符号指向薄膜内部。
    mu = np.sqrt(rng.random(N)) * (1 if side == "left" else -1)
    alive = np.ones(N, dtype=bool)
    residence = np.zeros(cells)
    transmitted = 0
    flight_time = 0.0
    while alive.any():
        ids = np.flatnonzero(alive)
        start = z[ids]
        vz = v*mu[ids]
        wall_time = np.full(ids.size, np.inf)
        positive, negative = vz > 0, vz < 0
        wall_time[positive] = (L-start[positive])/vz[positive]
        wall_time[negative] = -start[negative]/vz[negative]
        # Время до рассеяния имеет экспоненциальное распределение со средним τ.
        # 散射等待时间服从均值为 τ 的指数分布。
        free_time = rng.exponential(RELAXATION_TIME, size=ids.size)
        exits = wall_time <= free_time
        duration = np.minimum(free_time, wall_time)
        end = np.clip(start+vz*duration, 0, L)
        lo, hi = np.minimum(start,end)[:,None], np.maximum(start,end)[:,None]
        overlap = np.maximum(0, np.minimum(hi,edges[1:])-np.maximum(lo,edges[:-1]))
        moving = vz != 0
        # Время в ячейке = длина пересечения участка полёта / |v_z|.
        # 粒子在网格内的停留时间 = 飞行段与网格的重叠长度 / |v_z|。
        residence += (overlap[moving]/np.abs(vz[moving,None])).sum(axis=0)
        # При v_z = 0 всё время полёта относится к текущей ячейке.
        # 当 v_z = 0 时，整段飞行时间都计入当前网格。
        if (~moving).any():
            indices = np.clip(np.searchsorted(edges,start[~moving],side="right")-1,0,cells-1)
            np.add.at(residence, indices, duration[~moving])
        flight_time += duration.sum()
        target = vz > 0 if side == "left" else vz < 0
        transmitted += np.count_nonzero(exits & target)
        z[ids] = end
        alive[ids[exits]] = False
        # После рассеяния μ равномерен на [−1, 1]: направление изотропно в 3D.
        # 散射后 μ 在 [−1, 1] 上均匀分布，对应三维各向同性方向。
        mu[ids[~exits]] = 2*rng.random(np.count_nonzero(~exits))-1
    # Проверяем, что разбиение на ячейки сохраняет полное время полёта.
    # 检查网格划分前后的总飞行时间一致。
    assert np.isclose(residence.sum(), flight_time, rtol=1e-10, atol=1e-20)
    return transmitted, residence

n_lr, time_l = histories("left", rng)
n_rl, time_r = histories("right", rng)
# Мощность испускания отклонённой энергии может быть отрицательной у холодного термостата.
# 偏差能量发射率在冷端可以为负。
JL, JR = C*v*(TL-T0)/4, C*v*(TR-T0)/4
# Веса траекторий и времена пребывания дают отклонение температуры от T0.
# 利用轨迹权重与停留时间，计算相对于 T0 的温度偏差。
T = T0+(JL/N*time_l+JR/N*time_r)/(C*dz)
# Считаем чистый поток по вероятностям прохождения между термостатами.
# 根据两热库之间的穿透概率计算净热流。
q = JL*n_lr/N-JR*n_rl/N
k_eff = q*L/(TL-TR)
k_bulk = C*v*mean_free_path/3
print("L -> R, R -> L:", n_lr, n_rl)
print("q (МВт/м²), κ_eff, κ_bulk (Вт/(м·К)):", q/1e6, k_eff, k_bulk)
assert 0 < n_lr < N and 0 < n_rl < N
assert 0 < k_eff < k_bulk

### Температура и эффективная теплопроводность
В расчёте слайдов: 11105 и 11100 прохождений, κ_eff≈114.237 W/(m·K). Порядок вызовов генератора влияет на случайную реализацию.


In [ ]:
# Координаты центров ячеек переводим из метров в нанометры только для графика.
# 仅在绘图时，将网格中心坐标从米转换为纳米。
centers = (edges[:-1]+edges[1:])/2
plt.plot(centers/1e-9, T, "o-", label="Температура ячейки")
plt.plot([0,L/1e-9],[TL,TR],"s",label="Термостаты")
plt.xlabel("z, нм")
plt.ylabel("T, K")
plt.legend()
plt.show()